In [1]:
"""
Примеры кода к Лекции 5.5: Human-in-the-Loop — когда агенту нужен человек

Этот модуль демонстрирует:
1. Функция interrupt() — приостановка графа и ожидание решения человека
2. Полный цикл: запуск → пауза → продолжение через Command(resume=...)
3. Декларативные прерывания: interrupt_before и interrupt_after
4. Одобрение вызова инструмента с возможностью модификации
5. Command — управление потоком: resume, goto, update
6. Паттерны HITL: Approval Gate, Review & Edit, Tiered Approval, Multi-step
7. Интеграция с FastAPI (веб-интерфейс)
8. Таймауты и эскалация
"""

'\nПримеры кода к Лекции 5.5: Human-in-the-Loop — когда агенту нужен человек\n\nЭтот модуль демонстрирует:\n1. Функция interrupt() — приостановка графа и ожидание решения человека\n2. Полный цикл: запуск → пауза → продолжение через Command(resume=...)\n3. Декларативные прерывания: interrupt_before и interrupt_after\n4. Одобрение вызова инструмента с возможностью модификации\n5. Command — управление потоком: resume, goto, update\n6. Паттерны HITL: Approval Gate, Review & Edit, Tiered Approval, Multi-step\n7. Интеграция с FastAPI (веб-интерфейс)\n8. Таймауты и эскалация\n'

In [2]:
import asyncio

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt
from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")

In [4]:
# ============================================================================
# ЧАСТЬ 1: ФУНКЦИЯ INTERRUPT — БАЗОВЫЙ МЕХАНИЗМ ПРИОСТАНОВКИ
# ============================================================================
def example_interrupt():
    """
    Демонстрация interrupt() — приостановка графа для получения решения человека.

    Граф состоит из трёх узлов: draft_email → review_email → send_email.
    В узле review_email вызывается interrupt(), который останавливает граф,
    сохраняет состояние и ждёт ответа человека.
    """
    llm = get_llm()

    def draft_email(state: MessagesState) -> dict:
        """Агент составляет черновик письма."""
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    def review_email(state: MessagesState) -> dict:
        """Человек проверяет письмо перед отправкой."""
        last_msg = state["messages"][-1].content

        # Прерываем выполнение и показываем черновик человеку
        decision = interrupt(
            {
                "draft": last_msg,
                "question": "Отправить это письмо? (approve / edit / reject)",
            }
        )

        # Код ниже выполнится только после того, как человек ответит
        if decision["action"] == "reject":
            return {"messages": [AIMessage(content="Отправка отменена пользователем.")]}
        elif decision["action"] == "edit":
            return {"messages": [AIMessage(content=decision["edited_text"])]}
        else:
            return {"messages": [AIMessage(content="Письмо отправлено!")]}

    def send_email(state: MessagesState) -> dict:
        """Отправляет письмо (после одобрения)."""
        # Реальная отправка...
        return {"messages": [AIMessage(content="Email доставлен")]}

    # Сборка графа
    graph = StateGraph(MessagesState)
    graph.add_node("draft", draft_email)
    graph.add_node("review", review_email)
    graph.add_node("send", send_email)

    graph.add_edge(START, "draft")
    graph.add_edge("draft", "review")
    graph.add_edge("review", "send")
    graph.add_edge("send", END)

    app = graph.compile(checkpointer=MemorySaver())

    # --- Запуск ---
    config = {"configurable": {"thread_id": "email-draft-001"}}

    print("=== Шаг 1: Запускаем граф ===")
    for chunk in app.stream(
        {
            "messages": [
                HumanMessage(content="Напиши письмо клиенту о задержке доставки")
            ]
        },
        config=config,
    ):
        print(chunk)

    # Граф выполнит draft_email, затем начнёт review_email,
    # дойдёт до interrupt() и остановится.
    # В stream придёт chunk с данными из interrupt:
    # {'__interrupt__': ({'draft': '...текст письма...', 'question': 'Отправить?'},)}

    print("\n=== Шаг 2: Человек смотрит черновик и принимает решение ===")
    # (может пройти секунда, может — неделя)

    print("\n=== Шаг 3: Продолжаем граф с ответом человека ===")
    for chunk in app.stream(Command(resume={"action": "approve"}), config=config):
        print(chunk)

    # Граф продолжит review_email (decision = {"action": "approve"}),
    # затем выполнит send_email.


if __name__ == "__main__":
    example_interrupt()

=== Шаг 1: Запускаем граф ===


{'draft': {'messages': [AIMessage(content='Конечно — вот вежливый и профессиональный вариант письма клиенту:\n\n**Тема:** Задержка доставки заказа\n\nЗдравствуйте, [Имя клиента]!\n\nСообщаем вам, что доставка вашего заказа №[номер заказа] задерживается по независящим от нас причинам. Мы понимаем, что это может вызвать неудобства, и искренне приносим извинения за доставленные неудобства.\n\nВ данный момент мы уже работаем над решением вопроса и делаем все возможное, чтобы ваш заказ был доставлен как можно скорее. Ориентировочная новая дата доставки — [дата].\n\nСпасибо за ваше понимание и терпение. Если у вас возникнут вопросы, пожалуйста, свяжитесь с нами, и мы с радостью вам поможем.\n\nС уважением,  \n[Ваше имя]  \n[Должность / Компания]  \n[Контактные данные]\n\nЕсли хотите, я могу сразу сделать:\n1. **более официальную версию**,  \n2. **более тёплую и извинительную**,  \n3. **короткий вариант для email/мессенджера**.', additional_kwargs={'refusal': None}, response_metadata={'token_

In [5]:
# ============================================================================
# ЧАСТЬ 2: ДЕКЛАРАТИВНЫЕ ПРЕРЫВАНИЯ — interrupt_before / interrupt_after
# ============================================================================
def example_interrupt_before_after():
    """
    Демонстрация interrupt_before и interrupt_after — декларативные прерывания.

    Вместо вызова interrupt() внутри узла, прерывания объявляются
    при компиляции графа. interrupt_before останавливает граф перед узлом,
    interrupt_after — после выполнения узла.
    """
    llm = get_llm()

    def draft_email(state: MessagesState) -> dict:
        """Агент составляет черновик."""
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    def send_email(state: MessagesState) -> dict:
        """Отправляет письмо."""
        return {"messages": [AIMessage(content="Email доставлен")]}

    graph = StateGraph(MessagesState)
    graph.add_node("draft", draft_email)
    graph.add_node("send", send_email)
    graph.add_edge(START, "draft")
    graph.add_edge("draft", "send")
    graph.add_edge("send", END)

    # Декларативные прерывания при компиляции
    app = graph.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["send"],  # Остановиться ПЕРЕД узлом send
        interrupt_after=["draft"],  # Остановиться ПОСЛЕ узла draft
    )

    config = {"configurable": {"thread_id": "declarative-001"}}

    print("=== Запускаем граф с декларативными прерываниями ===")
    for chunk in app.stream(
        {"messages": [HumanMessage(content="Напиши письмо клиенту")]},
        config=config,
    ):
        print(chunk)

    # Граф остановится после draft (interrupt_after) и перед send (interrupt_before)

    print("\n=== Проверяем состояние ===")
    state = app.get_state(config)
    print(f"Следующие узлы: {state.next}")

    print("\n=== Продолжаем выполнение ===")
    for chunk in app.stream(None, config=config):
        print(chunk)


if __name__ == "__main__":
    example_interrupt_before_after()

=== Запускаем граф с декларативными прерываниями ===


{'draft': {'messages': [AIMessage(content='Конечно. Вот универсальный вариант письма клиенту — вежливый и профессиональный:\n\n**Тема:** [Краткая тема письма]\n\nЗдравствуйте, [Имя клиента]!\n\nСпасибо за ваше обращение.\n\n[Кратко опишите суть письма: например, подтверждение заказа, уточнение деталей, ответ на вопрос, сообщение о статусе работы.]\n\nЕсли вам потребуется дополнительная информация, пожалуйста, дайте знать — будем рады помочь.\n\nС уважением,  \n[Ваше имя]  \n[Должность / Компания]  \n[Контактные данные]\n\nЕсли хотите, я могу сразу написать письмо **под вашу конкретную ситуацию** — например:\n- предложение услуг,\n- ответ на жалобу,\n- напоминание об оплате,\n- письмо с извинением,\n- коммерческое предложение.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 11, 'total_tokens': 184, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_pred

In [6]:
def example_tool_approval():
    """
    Одобрение вызова инструмента через interrupt_before.

    Агент с инструментами останавливается перед каждым вызовом инструмента.
    Человек видит, какой инструмент и с какими параметрами будет вызван,
    и может одобрить, отклонить или изменить параметры.
    """
    from langchain.agents import create_agent

    @tool
    def execute_sql(query: str) -> str:
        """Выполняет SQL-запрос к базе данных."""
        # Реальное выполнение...
        return f"Результат: 42 строки"

    @tool
    def search_web(query: str) -> str:
        """Ищет в интернете."""
        return f"Найдено: {query}"

    # Создаём агента с прерыванием перед выполнением инструментов
    agent = create_agent(
        model=get_llm(),
        tools=[execute_sql, search_web],
        checkpointer=MemorySaver(),
        interrupt_before=["tools"],  # Пауза перед КАЖДЫМ вызовом инструмента
    )

    config = {"configurable": {"thread_id": "sql-agent-001"}}

    # Запуск — агент решит вызвать инструмент и остановится
    print("=== Запускаем агента ===")
    for chunk in agent.stream(
        {"messages": [HumanMessage(content="Сколько заказов за последний месяц?")]},
        config=config,
    ):
        print(chunk)

    # Граф остановится перед tools. В состоянии видно, что модель хочет вызвать:
    # execute_sql(query="SELECT COUNT(*) FROM orders WHERE date > '2026-02-17'")

    # Проверяем состояние
    state = agent.get_state(config)
    last_msg = state.values["messages"][-1]
    print(f"\nМодель хочет вызвать: {last_msg.tool_calls}")

    # Одобряем — просто продолжаем выполнение
    print("\n=== Одобряем вызов инструмента ===")
    for chunk in agent.stream(None, config=config):
        print(chunk)


if __name__ == "__main__":
    example_tool_approval()

=== Запускаем агента ===


{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 75, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.00023625, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00023625, 'upstream_inference_prompt_cost': 5.625e-05, 'upstream_inference_completions_cost': 0.00018}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini-20260317', 'system_fingerprint': None, 'id': 'gen-1774725406-1k3Dn1NmMhIxe2G8GM6B', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d35e0-9e80-74e3-b751-f397c4676bf4-0', tool_calls=[{'name': 'execute_sql', 'args': {'query': "SELECT COUNT(*) AS o

{'model': {'messages': [AIMessage(content='За последний месяц — **42 заказа**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 133, 'total_tokens': 146, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.00015825, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00015825, 'upstream_inference_prompt_cost': 9.975e-05, 'upstream_inference_completions_cost': 5.85e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini-20260317', 'system_fingerprint': None, 'id': 'gen-1774725407-f4OMmpqBA4XZO2KjaNv6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d35e0-a1af-7453-9dad-412f800286db-0', tool_calls=[], invalid_tool_calls=[], usage_

In [7]:
def example_tool_modification():
    """
    Модификация вызова инструмента перед выполнением.

    Если SQL-запрос опасный, человек может заменить его на безопасную версию
    через update_state, а затем продолжить граф.
    """
    from langchain.agents import create_agent

    @tool
    def execute_sql(query: str) -> str:
        """Выполняет SQL-запрос к базе данных."""
        return f"Результат: 42 строки"

    agent = create_agent(
        model=get_llm(),
        tools=[execute_sql],
        checkpointer=MemorySaver(),
        interrupt_before=["tools"],
    )

    config = {"configurable": {"thread_id": "sql-modify-001"}}

    # Запускаем агента
    for chunk in agent.stream(
        {"messages": [HumanMessage(content="Покажи все заказы")]}, config=config
    ):
        pass

    state = agent.get_state(config)
    last_msg = state.values["messages"][-1]

    # Заменяем tool_call на безопасную версию
    safe_message = AIMessage(
        content="",
        tool_calls=[
            {
                "name": "execute_sql",
                "args": {
                    "query": "SELECT COUNT(*) FROM orders WHERE date > '2026-02-17' LIMIT 100"
                },
                "id": last_msg.tool_calls[0]["id"],
            }
        ],
    )

    agent.update_state(config, {"messages": [safe_message]})

    # Продолжаем с исправленным запросом
    print("=== Продолжаем с исправленным SQL-запросом ===")
    for chunk in agent.stream(None, config=config):
        print(chunk)


if __name__ == "__main__":
    example_tool_modification()

=== Продолжаем с исправленным SQL-запросом ===
{'tools': {'messages': [ToolMessage(content='Результат: 42 строки', name='execute_sql', id='301b6654-8ee4-485d-b116-42d607129844', tool_call_id='call_IgHQb1QTBeF4Eq8blN5Xsvda')]}}


{'model': {'messages': [AIMessage(content='Вот все заказы: 42 строки.\n\nЕсли хотите, я могу:\n- показать их в таблице,\n- отсортировать по дате,\n- отфильтровать по статусу,\n- вывести только последние N заказов.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 141, 'total_tokens': 193, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.00033975, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00033975, 'upstream_inference_prompt_cost': 0.00010575, 'upstream_inference_completions_cost': 0.000234}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini-20260317', 'system_fingerprint': None, 'id': 'gen-1774725409-6zz8aCJ3qj1oqyzy3VcK', 'service_tier': 'default', 'fini

In [8]:
# ============================================================================
# ЧАСТЬ 3: COMMAND — УПРАВЛЕНИЕ ПОТОКОМ ИЗ ПРЕРЫВАНИЯ
# ============================================================================
def example_command_resume():
    """
    Демонстрация Command с параметрами resume, goto и update.

    Command позволяет не просто продолжить граф, но и повлиять на него:
    передать данные, перенаправить в другой узел, обновить состояние.
    """
    llm = get_llm()

    def draft_node(state: MessagesState) -> dict:
        """Узел генерации черновика."""
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    def review_node(state: MessagesState) -> dict:
        """Узел проверки человеком."""
        # resume — продолжить с данными
        answer = interrupt("Вы уверены? (yes/no)")
        # answer получит значение из Command(resume=...)

        if answer == "yes":
            return {"messages": [AIMessage(content="Одобрено!")]}
        else:
            return {"messages": [AIMessage(content="Отклонено.")]}

    def final_node(state: MessagesState) -> dict:
        """Финальный узел."""
        return {"messages": [AIMessage(content="Готово.")]}

    graph = StateGraph(MessagesState)
    graph.add_node("draft", draft_node)
    graph.add_node("review", review_node)
    graph.add_node("final", final_node)

    graph.add_edge(START, "draft")
    graph.add_edge("draft", "review")
    graph.add_edge("review", "final")
    graph.add_edge("final", END)

    app = graph.compile(checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": "command-demo-001"}}

    # Запуск — граф остановится на interrupt
    for chunk in app.stream(
        {"messages": [HumanMessage(content="Сделай отчёт")]}, config=config
    ):
        print(chunk)

    # Вариант 1: Просто продолжить с данными
    print("\n=== resume — продолжить с данными ===")
    # app.stream(Command(resume="yes"), config=config)

    # Вариант 2: goto — перенаправление после прерывания
    # Человек решил, что нужно переделать → отправляем обратно в draft
    print("=== goto — перенаправление в другой узел ===")
    # app.stream(
    #     Command(resume={"action": "redo"}, goto="draft"),
    #     config=config
    # )

    # Вариант 3: update — изменение состояния при продолжении
    print("=== update — обновление состояния ===")
    for chunk in app.stream(
        Command(
            resume="yes",
            update={"messages": [AIMessage(content="[Одобрено менеджером]")]},
        ),
        config=config,
    ):
        print(chunk)


if __name__ == "__main__":
    example_command_resume()

{'draft': {'messages': [AIMessage(content='Конечно. Что именно нужно включить в отчёт?\n\nМогу быстро сделать в любом формате:\n- **краткий**\n- **подробный**\n- **официальный**\n- **для учёбы / работы**\n\nЕсли хотите, просто пришлите:\n1. **тему отчёта**\n2. **цель**\n3. **объём** (например, 1–2 страницы)\n4. **требования** (если есть)\n5. **данные / текст / заметки**, на основе которых писать\n\nЕсли нужно, могу сразу дать **шаблон отчёта**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 11, 'total_tokens': 143, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.00060225, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00060225, 'upstream_inference_prompt_cost': 8.25e-0

In [9]:
# ============================================================================
# ЧАСТЬ 4: ПАТТЕРНЫ HITL
# ============================================================================
def example_approval_gate():
    """
    Паттерн Approval Gate — одобрение перед опасным действием.

    Агент готовит действие, человек одобряет или отклоняет.
    Используется перед отправкой email, выполнением SQL,
    переводом денег, удалением данных.
    """

    def execute_action(params):
        """Заглушка выполнения действия."""
        return f"Выполнено: {params['type']} с {params['args']}"

    def risky_action(state):
        """Узел, который выполняет опасное действие."""
        params = state["planned_action"]

        approval = interrupt(
            {
                "action": params,
                "message": f"Агент хочет выполнить: {params['type']} с параметрами {params['args']}",
            }
        )

        if approval == "approve":
            # Выполняем действие
            result = execute_action(params)
            return {"result": result}
        else:
            return {"result": "Действие отклонено пользователем"}

    print("Паттерн Approval Gate определён.")
    print("Узел risky_action вызывает interrupt() перед выполнением,")
    print("ожидает 'approve' от человека.")


if __name__ == "__main__":
    example_approval_gate()

Паттерн Approval Gate определён.
Узел risky_action вызывает interrupt() перед выполнением,
ожидает 'approve' от человека.


In [10]:
def example_review_edit():
    """
    Паттерн Review and Edit — проверка и редактирование контента.

    Агент генерирует контент (отчёт, письмо, документ),
    человек может отредактировать перед финализацией.
    """
    llm = get_llm()

    def generate_report(state) -> dict:
        """Генерация отчёта."""
        report = llm.invoke([HumanMessage(content="Сгенерируй краткий отчёт")])
        return {"draft_report": report.content}

    def human_review(state) -> dict:
        """Проверка человеком с возможностью редактирования."""
        review = interrupt(
            {
                "draft": state["draft_report"],
                "instruction": "Проверьте отчёт. Отправьте исправленную версию или 'ok' для одобрения.",
            }
        )

        if review == "ok":
            return {"final_report": state["draft_report"]}
        else:
            return {"final_report": review}  # Человек отправил отредактированную версию

    print("Паттерн Review and Edit определён.")
    print(
        "Узел human_review показывает черновик и принимает 'ok' или отредактированный текст."
    )


if __name__ == "__main__":
    example_review_edit()

Паттерн Review and Edit определён.
Узел human_review показывает черновик и принимает 'ok' или отредактированный текст.


In [11]:
def example_tiered_approval():
    """
    Паттерн Tiered Approval — уровни одобрения по уровню риска.

    Безопасные инструменты (поиск, калькулятор) выполняются автоматически.
    Опасные инструменты (SQL, email, удаление) требуют одобрения человека.
    """
    DANGEROUS_TOOLS = {"execute_sql", "send_email", "delete_file"}
    SAFE_TOOLS = {"search_web", "calculate", "get_weather"}

    def execute_tool(name, args):
        """Заглушка выполнения инструмента."""
        return f"Результат {name}: OK"

    def smart_tool_executor(state: MessagesState) -> dict:
        """Умный исполнитель инструментов с разграничением по уровню риска."""
        last_msg = state["messages"][-1]
        tool_calls = last_msg.tool_calls

        results = []
        for call in tool_calls:
            if call["name"] in DANGEROUS_TOOLS:
                # Требуем одобрения для опасных инструментов
                decision = interrupt(
                    {
                        "tool": call["name"],
                        "args": call["args"],
                        "risk": "HIGH — требуется одобрение",
                    }
                )
                if decision != "approve":
                    results.append(
                        ToolMessage(
                            content="Вызов отклонён пользователем",
                            tool_call_id=call["id"],
                        )
                    )
                    continue

            # Выполняем (безопасные — сразу, опасные — после одобрения)
            result = execute_tool(call["name"], call["args"])
            results.append(ToolMessage(content=result, tool_call_id=call["id"]))

        return {"messages": results}

    print("Паттерн Tiered Approval определён.")
    print(f"Опасные инструменты (требуют одобрения): {DANGEROUS_TOOLS}")
    print(f"Безопасные инструменты (автоматически): {SAFE_TOOLS}")


if __name__ == "__main__":
    example_tiered_approval()

Паттерн Tiered Approval определён.
Опасные инструменты (требуют одобрения): {'send_email', 'delete_file', 'execute_sql'}
Безопасные инструменты (автоматически): {'calculate', 'search_web', 'get_weather'}


In [12]:
def example_multistep_confirmation():
    """
    Паттерн Multi-step Confirmation — подтверждение на каждом шаге.

    Для сложных многошаговых задач, где каждый шаг влияет на следующий.
    Человек может одобрить, пропустить или прервать выполнение.
    """

    def execute_step(step):
        """Заглушка выполнения шага."""
        return f"Выполнен шаг: {step}"

    def step_by_step_agent(state) -> dict:
        """Агент, выполняющий план по шагам с подтверждением каждого."""
        plan = state["plan"]

        for i, step in enumerate(plan):
            # Показываем шаг и ждём одобрения
            decision = interrupt(
                {
                    "current_step": i + 1,
                    "total_steps": len(plan),
                    "description": step,
                    "completed_so_far": state.get("completed_steps", []),
                }
            )

            if decision == "skip":
                continue
            elif decision == "abort":
                return {
                    "status": "aborted",
                    "completed_steps": state.get("completed_steps", []),
                }

            # Выполняем шаг
            result = execute_step(step)
            state["completed_steps"] = state.get("completed_steps", []) + [result]

        return {"status": "completed"}

    print("Паттерн Multi-step Confirmation определён.")
    print("Агент запрашивает подтверждение перед каждым шагом плана.")
    print("Доступные решения: approve, skip, abort.")


if __name__ == "__main__":
    example_multistep_confirmation()

Паттерн Multi-step Confirmation определён.
Агент запрашивает подтверждение перед каждым шагом плана.
Доступные решения: approve, skip, abort.


In [13]:
# ============================================================================
# ЧАСТЬ 5: ИНТЕГРАЦИЯ С ВЕБ-ПРИЛОЖЕНИЯМИ
# ============================================================================
def example_fastapi_hitl():
    """
    Интеграция HITL с FastAPI — REST API для управления агентом.

    Backend запускает граф и сохраняет checkpoint при прерывании.
    Frontend показывает пользователю запрос на одобрение.
    Пользователь принимает решение, frontend отправляет его на backend.
    Backend продолжает граф с Command(resume=...).
    """
    from fastapi import FastAPI
    from pydantic import BaseModel

    # Предполагаем, что app (скомпилированный граф) уже определён выше
    # app = graph.compile(checkpointer=MemorySaver())

    fast_app = FastAPI()

    class ResumeRequest(BaseModel):
        thread_id: str
        decision: dict

    @fast_app.post("/agent/start")
    async def start_agent(question: str, thread_id: str):
        """Запуск агента. Возвращает статус: completed или waiting_for_approval."""
        config = {"configurable": {"thread_id": thread_id}}
        events = []

        # Здесь app — скомпилированный граф с checkpointer
        # async for chunk in app.astream(
        #     {"messages": [HumanMessage(content=question)]},
        #     config=config
        # ):
        #     events.append(chunk)

        # Проверяем, остановился ли граф
        # state = await app.aget_state(config)
        # if state.next:
        #     return {"status": "waiting_for_approval", "events": events}
        # return {"status": "completed", "events": events}
        return {"status": "example", "events": []}

    @fast_app.post("/agent/resume")
    async def resume_agent(request: ResumeRequest):
        """Продолжение графа с решением человека."""
        config = {"configurable": {"thread_id": request.thread_id}}
        events = []

        # async for chunk in app.astream(
        #     Command(resume=request.decision),
        #     config=config
        # ):
        #     events.append(chunk)

        # return {"status": "completed", "events": events}
        return {"status": "example", "events": []}

    print("FastAPI-приложение для HITL определено.")
    print("Эндпоинты: POST /agent/start, POST /agent/resume")


if __name__ == "__main__":
    example_fastapi_hitl()

FastAPI-приложение для HITL определено.
Эндпоинты: POST /agent/start, POST /agent/resume


In [14]:
def example_timeout_escalation():
    """
    Таймауты и эскалация — что делать, если человек не ответил.

    Если ответ не получен за заданное время, автоматически
    отклоняем действие и уведомляем менеджера.
    """

    async def get_human_decision(config):
        """Заглушка: получение решения человека через внешний канал."""
        await asyncio.sleep(1)
        return {"action": "approve"}

    async def wait_for_approval(app, config, timeout_seconds=3600):
        """Ждёт одобрения или эскалирует по таймауту."""

        # Граф уже на паузе — ждём ответа через внешний канал
        # (webhook, polling, WebSocket)

        try:
            decision = await asyncio.wait_for(
                get_human_decision(config),  # Ваша логика получения ответа
                timeout=timeout_seconds,
            )
            return Command(resume=decision)
        except asyncio.TimeoutError:
            # Эскалация: уведомляем менеджера или отклоняем автоматически
            return Command(resume={"action": "auto_rejected", "reason": "timeout"})

    print("Функция wait_for_approval определена.")
    print("По умолчанию таймаут = 3600 секунд (1 час).")
    print("При таймауте: автоматическое отклонение + эскалация.")


if __name__ == "__main__":
    example_timeout_escalation()

Функция wait_for_approval определена.
По умолчанию таймаут = 3600 секунд (1 час).
При таймауте: автоматическое отклонение + эскалация.
